# incident + runbook helper

In [ ]:
from typing import Annotated, Any, Dict, List, Optional, TypedDict

import os
import sqlite3
import uuid

import gradio as gr
import nest_asyncio
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain.agents import Tool
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field

In [ ]:
load_dotenv(override=True)

## structured eval

In [ ]:
class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="what was wrong or ok about the assistants answer")
    success_criteria_met: bool = Field(description="true if success criteria satisfied")
    user_input_needed: bool = Field(description="true if stuck or need user to clarify")

In [ ]:
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool

## tools: serper  + playwright

In [ ]:
nest_asyncio.apply()


browser = create_async_playwright_browser(headless=False)
pw_kit = PlayWrightBrowserToolkit.from_browser(async_browser=browser)
pw_tools = pw_kit.get_tools()

key = os.getenv("SERPER_API_KEY")
if not key:
    raise RuntimeError("set SERPER_API_KEY in .env")
serper = GoogleSerperAPIWrapper(serper_api_key=key)
tool_search = Tool(
    name="search",
    func=serper.run,
    description="web search for errors, docs, vendor pages",
)


def utc_stamp(_: str = "") -> str:
    """just stamps time for the incident log — custom tool like we did in class"""
    from datetime import datetime, timezone

    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")


tool_stamp = Tool(
    name="incident_time",
    func=utc_stamp,
    description="returns current utc time string for logging",
)

tools = [tool_search, tool_stamp] + pw_tools

In [ ]:
worker_llm = ChatOpenAI(model="gpt-4o-mini")
worker_llm_with_tools = worker_llm.bind_tools(tools)

eval_llm = ChatOpenAI(model="gpt-4o-mini")
eval_llm_structured = eval_llm.with_structured_output(EvaluatorOutput)

In [ ]:
def worker(state: State) -> Dict[str, Any]:
    sys = f"""You help with production incidents and runbooks.
Use search and browser tools when useful. Prefer official docs / status pages. Dont make up outages.
Success criteria for this turn:
{state['success_criteria']}
Reply with a clear answer or a clear Question: ... if you need the user.
"""
    if state.get("feedback_on_work"):
        sys += f"\nEvaluator said your last try wasnt enough: {state['feedback_on_work']}\nFix that."

    msgs = state["messages"]
    found = False
    for m in msgs:
        if isinstance(m, SystemMessage):
            m.content = sys
            found = True
    if not found:
        msgs = [SystemMessage(content=sys)] + list(msgs)

    out = worker_llm_with_tools.invoke(msgs)
    return {"messages": [out]}

In [ ]:
def worker_router(state: State) -> str:
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tools"
    return "evaluator"

In [ ]:
def _transcript(msgs: List[Any]) -> str:
    lines = []
    for m in msgs:
        if isinstance(m, HumanMessage):
            lines.append(f"user: {m.content}")
        elif isinstance(m, AIMessage):
            lines.append(f"assistant: {m.content or '[tool call]'}")
    return "\n".join(lines)

In [ ]:
def evaluator(state: State) -> Dict[str, Any]:
    last = state["messages"][-1].content
    sys = "You grade whether the assistant met the success criteria for an incident/runbook task."
    user = f"""conversation:
{_transcript(state['messages'])}

success criteria:
{state['success_criteria']}

last assistant text:
{last}
"""
    if state.get("feedback_on_work"):
        user += f"\nprevious feedback was: {state['feedback_on_work']}\n"

    res = eval_llm_structured.invoke(
        [SystemMessage(content=sys), HumanMessage(content=user)]
    )
    return {
        "messages": [
            {
                "role": "assistant",
                "content": f"(evaluator) {res.feedback}",
            }
        ],
        "feedback_on_work": res.feedback,
        "success_criteria_met": res.success_criteria_met,
        "user_input_needed": res.user_input_needed,
    }

In [ ]:
def after_eval(state: State) -> str:
    if state["success_criteria_met"] or state["user_input_needed"]:
        return "END"
    return "worker"

## graph + checkpoints 

In [ ]:
gb = StateGraph(State)
gb.add_node("worker", worker)
gb.add_node("tools", ToolNode(tools=tools))
gb.add_node("evaluator", evaluator)
gb.add_edge(START, "worker")
gb.add_conditional_edges(
    "worker", worker_router, {"tools": "tools", "evaluator": "evaluator"}
)
gb.add_edge("tools", "worker")
gb.add_conditional_edges(
    "evaluator", after_eval, {"worker": "worker", "END": END}
)


memory = MemorySaver()
graph = gb.compile(checkpointer=memory)

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
def make_thread_id():
    return str(uuid.uuid4())


async def run_turn(incident, criteria, history, thread):
    cfg = {"configurable": {"thread_id": thread}}
    state = {
        "messages": [HumanMessage(content=incident)],
        "success_criteria": criteria,
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False,
    }
    out = await graph.ainvoke(state, config=cfg)
    msgs = out["messages"]
    user = {"role": "user", "content": incident}
    answer = {"role": "assistant", "content": msgs[-2].content}
    fb = {"role": "assistant", "content": msgs[-1].content}
    h = history if history is not None else []
    return h + [user, answer, fb]


async def reset_all():
    return "", "", None, make_thread_id()

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## incident / runbook")
    tid = gr.State(make_thread_id())
    chat = gr.Chatbot(label="log", height=320, type="messages")
    inc = gr.Textbox(lines=3, label="what broke")
    crit = gr.Textbox(lines=2, label="done means")
    with gr.Row():
        b_go = gr.Button("go")
        b_rst = gr.Button("reset")
    b_go.click(run_turn, [inc, crit, chat, tid], [chat])
    inc.submit(run_turn, [inc, crit, chat, tid], [chat])
    crit.submit(run_turn, [inc, crit, chat, tid], [chat])
    b_rst.click(reset_all, [], [inc, crit, chat, tid])

demo.launch()